In [3]:
# pip install rapidfuzz

In [4]:
from google.colab import drive

try:
  drive.mount('/content/drive', force_remount=True) # Mencoba mount dan force remount jika perlu
except ValueError: # Menangkap error jika sudah ter-mount
  print("Google Drive sudah ter-mount.")

import sys
sys.path.append('/content/drive/Shareddrives/AFTERSALES EXTERNAL/SCRIPT/modules')  # Sesuaikan dengan path di Google Drive
import data_handler as dh
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import glob
import warnings
from google.colab import sheets
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1500)
np.seterr(all='raise')

import re
from collections import Counter
from itertools import chain
from rapidfuzz import process # Mengganti thefuzz dengan rapidfuzz sesuai skenario Anda
from IPython.display import display


Mounted at /content/drive


In [5]:
# =============================================================================
# MODULE: CONFIGURATION
# Simpan sebagai: config.py
# =============================================================================

class Config:
    # Google Sheet Sources
    SOURCE_SPREADSHEET = "Regenerate Import ELSA"
    SOURCE_WORKSHEET = "work_orders"

    # Dictionary Sources
    DICT_SPREADSHEET = "top_1000_keluhan_untuk_dilabeli"
    DICT_WORKSHEET = "top_keluhan"

    # Output Destinations
    OUTPUT_SHEET_ID = "1DiCivMEoFNDQxlaGIb66VdsVN9jsE-qkpQF2VJ28Xok"
    OUTPUT_WORKSHEET_DICT = "problem_dictionary"
    OUTPUT_WORKSHEET_RESULT = "wo_problem"

    # Thresholds
    FUZZY_THRESHOLD = 85

    # Regex Patterns
    # Membersihkan simbol selain huruf, angka, koma, spasi
    REGEX_CLEAN = r'[^a-z0-9,\s]'
    # Menyeragamkan delimiter menjadi titik koma (;)
    REGEX_SPLIT = r'(,\s*)|(\n)|(\s+(dan|&)\s+)|(\.\s*)'

In [6]:
# =============================================================================
# MODULE: UTILITIES / TEXT PROCESSOR
# Simpan sebagai: utils.py
# =============================================================================
import re

class TextProcessor:
    @staticmethod
    def clean_and_split(text):
        """Membersihkan teks dan memecahnya menjadi list standar."""
        if not isinstance(text, str):
            return []

        text_lower = text.lower()
        # Hapus simbol aneh (menggunakan pattern standar)
        text_cleaned = re.sub(r'[^a-z0-9,\s]', '', text_lower)
        # Ganti semua delimiter jadi ;
        text_std = re.sub(r'(,\s*)|(\n)|(\s+(dan|&)\s+)|(\.\s*)', ';', text_cleaned)
        # Split dan strip
        return [item.strip() for item in text_std.split(';') if item.strip()]

    @staticmethod
    def generate_problem_name(part, masalah, section, label_standar):
        """Membuat nama masalah yang user-friendly (Title Case)."""
        part = part.strip().capitalize() if part else ""
        masalah = masalah.strip().capitalize() if masalah else ""
        section = section.strip().lower() if section else ""

        # Handling khusus
        if masalah == 'Aus(twi)':
            masalah = 'Aus/TWI'

        if section == 'neutral':
            return f"{part} {masalah}"
        elif section in ['rear', 'front']:
            section_text = 'Depan' if section == 'front' else 'Belakang'
            return f"{part} {section_text} {masalah}"
        elif section in ['rear/front', 'left/right', 'all']:
            return f"Semua {part} {masalah}"
        else:
            # Fallback ke label standar jika format aneh
            return ' '.join([word.capitalize() for word in label_standar.split('_')])

In [7]:
# =============================================================================
# MODULE: DICTIONARY MANAGER
# Simpan sebagai: dictionary_manager.py
# =============================================================================
import pandas as pd
import data_handler as dh

class DictionaryManager:
    def __init__(self, config_class, text_processor_class):
        self.config = config_class
        self.text_processor = text_processor_class
        self.grouped_dictionary = None

    def load_and_build(self):
        """Memuat data mentah dari sheet dan mengubahnya menjadi format grouped."""
        print("\n[DictionaryManager] Memuat kamus manual...")
        df_raw = dh.load_gspread_data(self.config.DICT_SPREADSHEET, self.config.DICT_WORKSHEET)

        required_cols = ['raw_keluhan', 'part', 'masalah', 'label_standar', 'section']
        if not all(col in df_raw.columns for col in required_cols):
            raise ValueError(f"Kolom kamus tidak lengkap. Wajib ada: {required_cols}")

        df_clean = df_raw.dropna(subset=required_cols).copy()

        # Grouping: Menggabungkan raw_keluhan yang sama label-nya menjadi satu list
        self.grouped_dictionary = df_clean.groupby(['label_standar', 'section']).agg(
            part=('part', 'first'),
            masalah=('masalah', 'first'),
            raw_keluhan_list=('raw_keluhan', list)
        ).reset_index()

        # Generate Problem Name untuk setiap baris kamus
        self.grouped_dictionary['problem_name'] = self.grouped_dictionary.apply(
            lambda row: self.text_processor.generate_problem_name(
                row['part'], row['masalah'], row['section'], row['label_standar']
            ), axis=1
        )

        print(f"[DictionaryManager] Kamus siap dengan {len(self.grouped_dictionary)} kategori unik.")
        return self.grouped_dictionary

    def upload_dictionary(self):
        """Mengunggah versi kamus yang sudah diproses ke Google Sheet (Backup/View)."""
        if self.grouped_dictionary is None:
            print("Kamus belum dibuat, tidak bisa upload.")
            return

        print("[DictionaryManager] Mengunggah kamus terkelompok...")
        to_upload = self.grouped_dictionary.copy()
        # Convert list ke string agar bisa masuk cell excel
        to_upload['raw_keluhan_list'] = to_upload['raw_keluhan_list'].apply(
            lambda x: ', '.join(x) if isinstance(x, list) else x
        )
        dh.clean_and_upload_to_google_sheet(
            to_upload, self.config.OUTPUT_SHEET_ID, self.config.OUTPUT_WORKSHEET_DICT
        )

In [8]:
# =============================================================================
# MODULE: COMPLAINT NORMALIZER
# Simpan sebagai: normalizer.py
# =============================================================================
from rapidfuzz import process

class ComplaintNormalizer:
    def __init__(self, dictionary_df, fuzzy_threshold=85):
        self.dictionary_df = dictionary_df
        self.threshold = fuzzy_threshold

    def normalize(self, keluhan_text):
        """
        Input: String keluhan tunggal (misal: 'ban depan halus')
        Output: Dictionary object berisi detail standar
        """
        if not isinstance(keluhan_text, str) or self.dictionary_df is None:
            return None

        keluhan_clean = keluhan_text.strip().lower()
        best_match = None
        best_score = -1

        # Iterasi kamus
        for _, row in self.dictionary_df.iterrows():
            raw_list = [str(r).strip().lower() for r in row['raw_keluhan_list']]

            # A. Exact Match (Prioritas 1)
            if keluhan_clean in raw_list:
                return self._format_result(row, keluhan_clean)

            # B. Fuzzy Match (Prioritas 2)
            if raw_list:
                match, score, _ = process.extractOne(keluhan_clean, raw_list)
                if score > self.threshold and score > best_score:
                    best_score = score
                    best_match = self._format_result(row, match)

        return best_match

    def _format_result(self, row, raw_match):
        return {
            'category': row['label_standar'],
            'section': row['section'],
            'part': row['part'],
            'masalah': row['masalah'],
            'problem_name': row['problem_name'],
            'raw_match': raw_match
        }

In [9]:
# =============================================================================
# MODULE: ORCHESTRATOR (MAIN)
# Simpan sebagai: orchestrator.py
# =============================================================================
import pandas as pd
import data_handler as dh
from IPython.display import display

# CATATAN:
# Jika Anda menjalankan ini di satu notebook (all-in-one cell), class Config, TextProcessor,
# DictionaryManager, dan ComplaintNormalizer sudah harus didefinisikan sebelumnya.
# Jika file terpisah, uncomment import di bawah ini:
# from config import Config
# from utils import TextProcessor
# from dictionary_manager import DictionaryManager
# from normalizer import ComplaintNormalizer

class PipelineOrchestrator:
    def __init__(self):
        # Inisialisasi modul-modul
        # Asumsi class Config dan TextProcessor tersedia di scope global
        self.dict_manager = DictionaryManager(Config, TextProcessor)
        self.normalizer = None
        self.data = None

    def run(self):
        print("=== MEMULAI PIPELINE ANALISIS KELUHAN ===")

        # STEP 1: Persiapan Kamus
        kamus_df = self.dict_manager.load_and_build()
        self.dict_manager.upload_dictionary() # Opsional: Backup kamus
        self.normalizer = ComplaintNormalizer(kamus_df, Config.FUZZY_THRESHOLD)

        # STEP 2: Load Data Transaksi
        print("\n[Orchestrator] Memuat data Work Orders...")
        self.data = dh.load_gspread_data(Config.SOURCE_SPREADSHEET, Config.SOURCE_WORKSHEET)

        # STEP 3: Preprocessing (Cleaning & Splitting)
        print("[Orchestrator] Membersihkan dan memecah keluhan...")
        self.data['keluhan_list'] = self.data['customer_problems'].apply(TextProcessor.clean_and_split)

        # STEP 4: Normalisasi (Core Process)
        print("[Orchestrator] Menjalankan normalisasi (Mungkin butuh waktu)...")

        def process_row_list(keluhan_list):
            results = []
            for k in keluhan_list:
                norm = self.normalizer.normalize(k)
                if norm: results.append(norm)
            return results

        self.data['normalized_results'] = self.data['keluhan_list'].apply(process_row_list)

        # STEP 5: Post-Processing & Reporting
        self._generate_and_upload_report()
        print("\n=== PIPELINE SELESAI ===")

    def _generate_and_upload_report(self):
        print("\n[Orchestrator] Membuat laporan akhir...")

        # Explode data agar 1 baris = 1 masalah teridentifikasi
        df_exploded = self.data.explode('normalized_results').dropna(subset=['normalized_results'])

        # Ekstrak data dari dictionary hasil normalisasi
        keys_to_extract = ['category', 'section', 'part', 'problem_name', 'raw_match']
        for key in keys_to_extract:
            df_exploded[key] = df_exploded['normalized_results'].apply(lambda x: x.get(key))

        # Grouping kembali per Order ID
        final_report = df_exploded.groupby('order_id').agg(
            customer_problems=('customer_problems', 'first'),
            # Categorized problems tetap list of dict (untuk detail lengkap)
            categorized_problems=('normalized_results', list),

            # --- PENGGABUNGAN TEXT BIASA (COMMA SEPARATED) ---
            # Mengubah list problem_names menjadi satu string dipisah koma
            problem_names=('problem_name', lambda x: ', '.join([str(item) for item in x if item])),

            raw_matches=('raw_match', list)
        ).reset_index()

        # Membersihkan Data sebelum Upload (Handling NaN/Inf)
        final_report = final_report.astype(str).replace(['nan', 'inf', '-inf'], '')

        print("[Orchestrator] Mengunggah laporan ke Google Sheets...")
        dh.clean_and_upload_to_google_sheet(
            final_report, Config.OUTPUT_SHEET_ID, Config.OUTPUT_WORKSHEET_RESULT
        )
        display(final_report.head())

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    app = PipelineOrchestrator()
    app.run()

=== MEMULAI PIPELINE ANALISIS KELUHAN ===

[DictionaryManager] Memuat kamus manual...
[DictionaryManager] Kamus siap dengan 145 kategori unik.
[DictionaryManager] Mengunggah kamus terkelompok...
Worksheet 'problem_dictionary' telah dibersihkan.
Data berhasil diunggah (overwrite) ke worksheet 'problem_dictionary'.

[Orchestrator] Memuat data Work Orders...
[Orchestrator] Membersihkan dan memecah keluhan...
[Orchestrator] Menjalankan normalisasi (Mungkin butuh waktu)...

[Orchestrator] Membuat laporan akhir...
[Orchestrator] Mengunggah laporan ke Google Sheets...
Worksheet 'wo_problem' telah dibersihkan.
Data berhasil diunggah (overwrite) ke worksheet 'wo_problem'.


,order_id,customer_problems,categorized_problems,problem_names,raw_matches
0,WO-100283849900168319,Disc depan belakang \nKampas depan belakang,"[{'category': 'DISCBRAKE_TIPIS', 'section': 'r...","Semua Disc brake Tipis, Semua Rem Bunyi","['disc depan belakang', 'kampas depan belakang']"
1,WO-100284755869831187,Ganti ban belakang \nservice rutin berkala,"[{'category': 'BAN_AUS(TWI)', 'section': 'rear...","Ban Belakang Aus (twi), Pengecekan Unit","['ganti ban belakang', 'service rutin berkala']"
2,WO-100285489873033512,Ban bocor tidak visa ditambal,"[{'category': 'BAN_BOCOR', 'section': 'rear/fr...",Semua Ban Bocor,['ban bocor tidak bisa ditambal']
3,WO-100285762502794870,Kampas depan belakang \nganti ban belakang \nc...,"[{'category': 'REM_BUNYI', 'section': 'rear/fr...","Semua Rem Bunyi, Ban Belakang Aus (twi), Mesin...","['kampas depan belakang', 'ganti ban belakang'..."
4,WO-100286769135754990,Service rutin berkala \ndudukan plate belakang,"[{'category': 'CHECK_POIN', 'section': 'neutra...","Pengecekan Unit, Plat nomor Belakang Hilang","['service rutin berkala', 'plate belakang']"



=== PIPELINE SELESAI ===
